In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [91]:
portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

merged = pd.read_csv('../data/all_merged.csv')
promotion = pd.read_csv('../data/promotion_df.csv')
normal = pd.read_csv('../data/normal_flow_df.csv')
final_df = pd.read_csv('../data/promotion_df_final5.csv')

---

In [93]:
# 현재 코드 확인
normal_df_check = final_df[final_df['is_normal_flow'] == 1].copy()
print(f'정상흐름 행 수: {len(normal_df_check)}')
print(f'amount_from_received < difficulty: {(normal_df_check["amount_from_received"] < normal_df_check["difficulty"]).sum()}')

정상흐름 행 수: 23267
amount_from_received < difficulty: 76


In [94]:
weird_df = final_df[
    (final_df['is_normal_flow'] == 1) & 
    (final_df['amount_from_received'] < final_df['difficulty'])
].copy()

print(f'총 {len(weird_df)}건')
print()
print('=== offer_id별 건수 ===')
print(weird_df['offer_id'].value_counts())
print()
display(weird_df[[
    'person', 'offer_id', 'offer_cycle',
    'offer received', 'offer viewed', 'offer completed',
    'difficulty', 'amount_from_received', 'amount_from_viewed'
]].sort_values('amount_from_received').head(20))

총 76건

=== offer_id별 건수 ===
offer_id
discount_10_2_10    40
discount_7_3_7      15
discount_20_5_10    12
discount_10_2_7      9
Name: count, dtype: int64



,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,difficulty,amount_from_received,amount_from_viewed
5373,2ac20da42e704b4bae0a799018096259,discount_7_3_7,Discount_2,408.0,444.0,456.0,7,0.59,0.59
19419,95f37a4a6f8b4bf6b9f91d62db4457dc,discount_7_3_7,Discount_2,504.0,510.0,522.0,7,0.80,0.80
20389,9cbf831239914565aebb4495c0abecf8,discount_10_2_10,Discount_2,576.0,582.0,594.0,10,0.98,0.98
13210,65e5f48999544aedada20102086918f7,discount_10_2_10,Discount_3,576.0,630.0,636.0,10,1.12,1.12
24470,bc3694dd6a25450ca80f4d4a60719236,discount_7_3_7,Discount_2,576.0,576.0,618.0,7,1.58,1.58
30121,e898f8d0cc514692aee817765d1584e9,discount_10_2_10,Discount_3,576.0,576.0,642.0,10,1.63,1.63
1254,09e7e40eb89943458f4f79fe2e023dd6,discount_7_3_7,Discount_2,576.0,576.0,606.0,7,2.01,2.01
15815,7a4f2c4d9ddd4279a83ad86b4b5a81f2,discount_10_2_10,Discount_2,576.0,576.0,576.0,10,2.06,2.06
10066,4d81f9f887724401a9c7606db4afef01,discount_10_2_10,Discount_2,168.0,186.0,204.0,10,2.11,2.11
17458,86d1c0779345461fb1f5811627c93b3e,discount_10_2_10,Discount_2,576.0,582.0,588.0,10,2.21,2.21


In [42]:
# 전에 쓰던 promotion_df와 현재 final_df의 정상흐름 비교
promotion_orig = pd.read_csv('../data/promotion_df.csv')
print(f'원래 정상흐름(is_normal_flow==1): {promotion_orig["is_normal_flow"].sum()}')
print(f'현재 정상흐름(flow_type==1): {(final_df["flow_type"]==1).sum()}')

# offer_type별 비교
print()
print('=== 원래 ===')
print(promotion_orig[promotion_orig['is_normal_flow']==1]['offer_type'].value_counts())
print()
print('=== 현재 ===')
print(final_df[final_df['flow_type']==1]['offer_type'].value_counts())

원래 정상흐름(is_normal_flow==1): 23519
현재 정상흐름(flow_type==1): 23267

=== 원래 ===
offer_type
discount    12501
bogo        11018
Name: count, dtype: int64

=== 현재 ===
offer_type
discount    12326
bogo        10941
Name: count, dtype: int64


In [43]:
import ast

promotion_df = pd.read_csv('../data/promotion_df_final.csv')
transcript = pd.read_csv('../data/transcript.csv')

# transcript 전처리
transcript['value'] = transcript['value'].apply(ast.literal_eval)
value_df = pd.DataFrame(transcript['value'].tolist())
transcript = pd.concat([transcript, value_df], axis=1)
transcript['offer_id'] = transcript['offer_id'].fillna(transcript['offer id'])
transcript = transcript.drop(columns=['offer id', 'value'], errors='ignore')

# transactions_df 만들기
transactions_df = transcript[transcript['event'] == 'transaction'][['person', 'time', 'amount']].copy()
print(f'transactions_df 행 수: {len(transactions_df)}')

# tx_dict
tx_dict = {
    person: group.sort_values('time')
    for person, group in transactions_df.groupby('person')
}
print(f'tx_dict person 수: {len(tx_dict)}')

transactions_df 행 수: 138953
tx_dict person 수: 16578


In [44]:
# 정상흐름만 필터링
normal_df = promotion_df[promotion_df['is_normal_flow'] == 1].copy()
print(f'정상흐름 행 수: {len(normal_df)}')

# cycle 번호 추출
normal_df['cycle_num'] = normal_df['offer_cycle'].str.extract(r'_(\d+)$').astype(int)

# first_received 계산
first_received = (
    promotion_df.groupby(['person', 'offer_id'])['offer received']
    .min()
    .reset_index()
    .rename(columns={'offer received': 'first_received'})
)
normal_df = normal_df.merge(first_received, on=['person', 'offer_id'], how='left')

results = []
for _, row in normal_df.iterrows():
    person_tx = tx_dict.get(row['person'], pd.DataFrame(columns=['time', 'amount']))
    received_start = row['first_received'] if row['cycle_num'] == 1 else row['offer received']

    amt_received = person_tx[
        (person_tx['time'] >= received_start) &
        (person_tx['time'] <= row['offer completed'])
    ]['amount'].sum()

    amt_viewed = person_tx[
        (person_tx['time'] >= row['offer viewed']) &
        (person_tx['time'] <= row['offer completed'])
    ]['amount'].sum()

    results.append({
        'amount_from_received': amt_received,
        'amount_from_viewed': amt_viewed
    })

result_df = pd.DataFrame(results, index=normal_df.index)
normal_df['amount_from_received'] = result_df['amount_from_received']
normal_df['amount_from_viewed'] = result_df['amount_from_viewed']
normal_df = normal_df.drop(columns=['first_received', 'cycle_num'], errors='ignore')

print(f"amount_from_received < difficulty: {(normal_df['amount_from_received'] < normal_df['difficulty']).sum():,}")
print(f"amount_from_viewed < difficulty: {(normal_df['amount_from_viewed'] < normal_df['difficulty']).sum():,}")

정상흐름 행 수: 23267
amount_from_received < difficulty: 76
amount_from_viewed < difficulty: 684


In [45]:
promotion_orig = pd.read_csv('../data/promotion_df.csv')

# 원래에는 있는데 새 final_df에는 없는 정상흐름 케이스
orig_normal = promotion_orig[promotion_orig['is_normal_flow'] == 1][['person', 'offer_id', 'offer_cycle']].drop_duplicates()
new_normal = final_df[final_df['is_normal_flow'] == 1][['person', 'offer_id', 'offer_cycle']].drop_duplicates()

# 차집합
missing = orig_normal.merge(new_normal, on=['person', 'offer_id', 'offer_cycle'], how='left', indicator=True)
missing = missing[missing['_merge'] == 'left_only']
print(f'원래엔 있는데 새 final_df에 없는 정상흐름: {len(missing)}건')
print(missing['offer_id'].value_counts())

원래엔 있는데 새 final_df에 없는 정상흐름: 0건
Series([], Name: count, dtype: int64)


In [46]:
# final_df에 이미 amount_from_received가 있는지 확인
print(promotion_df.columns.tolist())
print(promotion_df['amount_from_received'].notna().sum())

['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed', 'offer_type', 'difficulty', 'reward', 'duration', 'web', 'email', 'mobile', 'social', 'gender', 'age', 'became_member_on', 'income', 'amount', 'is_received', 'is_viewed', 'is_completed', 'is_normal_flow', 'is_deduplicated', 'flow_type', 'amount_from_received', 'amount_from_viewed']
23267


In [47]:
import pandas as pd

promotion_df = pd.read_csv('../data/promotion_df_final.csv')

weird_df = promotion_df[
    (promotion_df['is_normal_flow'] == 1) & 
    (promotion_df['amount_from_received'] < promotion_df['difficulty'])
].copy()

print(f'총 {len(weird_df)}건')
print()
print('=== offer_id별 건수 ===')
print(weird_df['offer_id'].value_counts())

총 626건

=== offer_id별 건수 ===
offer_id
discount_20_5_10    224
discount_10_2_10    142
bogo_10_10_5         95
bogo_10_10_7         75
discount_10_2_7      65
discount_7_3_7       23
bogo_5_5_7            1
bogo_5_5_5            1
Name: count, dtype: int64


In [48]:
# bogo 케이스 하나만 직접 확인
sample = weird_df[weird_df['offer_id'].str.contains('bogo')].iloc[0]
print(sample[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer completed', 'difficulty', 'amount_from_received']])

# 해당 person의 트랜잭션 확인
person_tx = tx_dict.get(sample['person'])
print(person_tx[
    (person_tx['time'] >= sample['offer received']) &
    (person_tx['time'] <= sample['offer completed'])
])

person                  00bc42a62f884b41a13cc595856cf7c3
offer_id                                    bogo_10_10_7
offer_cycle                                       Bogo_1
offer received                                     336.0
offer completed                                    390.0
difficulty                                            10
amount_from_received                                7.35
Name: 62, dtype: object
                                  person  time  amount
146542  00bc42a62f884b41a13cc595856cf7c3   390   17.03


In [49]:
person_id = '00bc42a62f884b41a13cc595856cf7c3'
person_tx = tx_dict.get(person_id)
print(person_tx[(person_tx['time'] >= 336) & (person_tx['time'] <= 390)])

                                  person  time  amount
146542  00bc42a62f884b41a13cc595856cf7c3   390   17.03


In [50]:
print(f'transcript 전체 행 수: {len(transcript)}')
print(transcript['event'].value_counts())
print()
print(transcript[transcript['event'] == 'transaction'][['person', 'time', 'amount']].head())
print(f'transaction 행 수: {len(transcript[transcript["event"] == "transaction"])}')

transcript 전체 행 수: 306534
event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

                                 person  time  amount
12654  02c083884c7d45b39cc68e1314fec56c     0    0.83
12657  9fa9ae8f57894cc9a3b8a9bbe0fc1b2f     0   34.56
12659  54890f68699049c2a04d415abc25e717     0   13.23
12670  b2f1cd155b864803ad8334cdf13c4bd2     0   19.51
12671  fe97aa22dd3e48c8b143116a8403dd52     0   18.97
transaction 행 수: 138953


In [51]:
# 626건 중 하나 뽑아서 확인
normal_df = promotion_df[promotion_df['is_normal_flow'] == 1].copy()

weird_df = normal_df[normal_df['amount_from_received'] < normal_df['difficulty']].copy()
print(f'총 {len(weird_df)}건')

# 첫 번째 케이스 확인
sample = weird_df.iloc[0]
print(sample[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer completed', 'difficulty', 'amount_from_received']])

# 해당 person의 트랜잭션 확인
person_tx = tx_dict.get(sample['person'], pd.DataFrame(columns=['time', 'amount']))
mask = (person_tx['time'] >= sample['offer received']) & (person_tx['time'] <= sample['offer completed'])
print(f'\nreceived~completed 사이 트랜잭션:')
print(person_tx[mask])
print(f'합계: {person_tx[mask]["amount"].sum()}')

총 626건
person                  0011e0d4e6b944f998e987f904e8c1e5
offer_id                                discount_20_5_10
offer_cycle                                   Discount_1
offer received                                     408.0
offer completed                                    576.0
difficulty                                            20
amount_from_received                               10.51
Name: 5, dtype: object

received~completed 사이 트랜잭션:
                                  person  time  amount
258979  0011e0d4e6b944f998e987f904e8c1e5   576   22.05
합계: 22.05


In [52]:
person_id = '0011e0d4e6b944f998e987f904e8c1e5'

print('=== transcript 전체 트랜잭션 ===')
print(transactions_df[transactions_df['person'] == person_id].sort_values('time'))

print()
print('=== tx_dict 트랜잭션 ===')
print(tx_dict.get(person_id))

=== transcript 전체 트랜잭션 ===
                                  person  time  amount
47805   0011e0d4e6b944f998e987f904e8c1e5   132   13.49
95421   0011e0d4e6b944f998e987f904e8c1e5   252   11.93
258979  0011e0d4e6b944f998e987f904e8c1e5   576   22.05
288294  0011e0d4e6b944f998e987f904e8c1e5   642   23.03
291925  0011e0d4e6b944f998e987f904e8c1e5   654    8.96

=== tx_dict 트랜잭션 ===
                                  person  time  amount
47805   0011e0d4e6b944f998e987f904e8c1e5   132   13.49
95421   0011e0d4e6b944f998e987f904e8c1e5   252   11.93
258979  0011e0d4e6b944f998e987f904e8c1e5   576   22.05
288294  0011e0d4e6b944f998e987f904e8c1e5   642   23.03
291925  0011e0d4e6b944f998e987f904e8c1e5   654    8.96


In [53]:
normal_df = promotion_df[promotion_df['is_normal_flow'] == 1].copy()

weird_df = normal_df[normal_df['amount_from_received'] < normal_df['difficulty']].copy()
print(f'총 {len(weird_df)}건')

# 첫 번째 케이스 확인
sample = weird_df.iloc[0]
print(sample[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer completed', 'difficulty', 'amount_from_received']])

# 해당 구간 트랜잭션 확인
person_tx = tx_dict.get(sample['person'], pd.DataFrame(columns=['time', 'amount']))
mask = (person_tx['time'] >= sample['offer received']) & (person_tx['time'] <= sample['offer completed'])
print(f'\n해당 구간 트랜잭션:')
print(person_tx[mask])
print(f'합계: {person_tx[mask]["amount"].sum()}')

총 626건
person                  0011e0d4e6b944f998e987f904e8c1e5
offer_id                                discount_20_5_10
offer_cycle                                   Discount_1
offer received                                     408.0
offer completed                                    576.0
difficulty                                            20
amount_from_received                               10.51
Name: 5, dtype: object

해당 구간 트랜잭션:
                                  person  time  amount
258979  0011e0d4e6b944f998e987f904e8c1e5   576   22.05
합계: 22.05


In [54]:
# 현재 tx_dict에 들어간 transactions_df가 뭔지 확인
print(f'transactions_df 행 수: {len(transactions_df)}')
print(f'transactions_df time 범위: {transactions_df["time"].min()} ~ {transactions_df["time"].max()}')

transactions_df 행 수: 138953
transactions_df time 범위: 0 ~ 714


In [55]:
# 위 코드 그대로 돌리고
print(f'amount_from_received < difficulty: {(normal_df["amount_from_received"] < normal_df["difficulty"]).sum()}')

# 626건 중 어떤 케이스인지 확인
weird_df = normal_df[normal_df['amount_from_received'] < normal_df['difficulty']].copy()
print(weird_df['offer_cycle'].str.extract(r'_(\d+)$')[0].value_counts())

amount_from_received < difficulty: 626
0
1    514
2    101
3     11
Name: count, dtype: int64


In [95]:
final_df.head()

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,flow_type,amount_from_received,amount_from_viewed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,72000.0,8.57,1,1,1,0,1,0,8.57,0.00
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,72000.0,14.11,1,1,1,0,1,0,14.11,0.00
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,72000.0,10.27,1,0,1,0,1,2,10.27,0.00
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,57000.0,11.93,1,1,1,1,1,1,11.93,11.93
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,57000.0,22.05,1,1,1,1,1,1,22.05,22.05
